# MODULE 1 - DOCUMENT LOADING

LEARNING OBJECTIVES:
1. Understand PDF structure and how to extract text
2. Learn about different PDF loaders and their trade-offs
3. Handle multiple PDF files in a directory
4. Extract metadata from PDFs (page numbers, file names)
5. Deal with common PDF loading issues

In [ ]:
# installing required libraries
!pip install pypdf langchain langchain-community -q

In [ ]:
# importing libraries
import os
from pathlib import Path
import urllib.request
from typing import List, Dict

# PyPDF for basic PDF reading
from pypdf import PdfReader

# LangChain's PDF loader (built on top of pypdf)
from langchain_community.document_loaders import PyPDFLoader


pypdf: Low-level library that gives us direct control over PDF parsing

LangChain's PyPDFLoader: High-level wrapper that handles documents nicely
LangChain creates "Document" objects with:
* page_content: The actual text
* metadata: Information about the document (page number, source, etc.)



In [ ]:
# downloading a sample pdf file

# creating a directory for my pdf
os.makedirs("sample_pdfs", exist_ok=True)

# Using a simple research paper PDF from arXiv
sample_pdfs = {
    "attention_paper.pdf": "https://arxiv.org/pdf/1706.03762.pdf",  # "Attention is All You Need" paper
}

for filename, url in sample_pdfs.items():
    filepath = f"sample_pdfs/{filename}"
    if not os.path.exists(filepath):
        print(f"Downloading: {filename}...")
        try:
            urllib.request.urlretrieve(url, filepath)
            print(f"Downloaded: {filename}")
        except Exception as e:
            print(f"Failed to download {filename}: {e}")
    else:
        print(f"Already exists: {filename}")

print("\nPDF files ready!\n")

Already exists: attention_paper.pdf

PDF files ready!



In [ ]:
# method 1 - low level approach , direct control and access to metadata
def load_pdf_with_pypdf(pdf_path: str) -> Dict:
  """
  this method gives us fine grained control but requires more manual work
  returns: dictionary with text and metadata
  """
  reader = PdfReader(pdf_path)
  # extracting metadata
  metadata = {
        "filename": os.path.basename(pdf_path),
        "num_pages": len(reader.pages),
        "author": reader.metadata.get('/Author', 'Unknown'),
        "title": reader.metadata.get('/Title', 'Unknown'),
    }

  #extracting text form each page
  pages_text = []
  for page_num, page in enumerate(reader.pages, start=1):
    text = page.extract_text()
    pages_text.append({
        "page_number": page_num,
            "text": text,
            "char_count": len(text)
    })

  return {
      "metadata": metadata,
      "pages": pages_text
  }

In [ ]:
# testing the method 1
pdf_path = "sample_pdfs/attention_paper.pdf"
if os.path.exists(pdf_path):
    result = load_pdf_with_pypdf(pdf_path)

    print(f"📄 File: {result['metadata']['filename']}")
    print(f"📊 Total Pages: {result['metadata']['num_pages']}")
    print(f"✍️  Author: {result['metadata']['author']}")
    print(f"📖 Title: {result['metadata']['title']}")
    print(f"\n📝 First page preview (first 300 characters):")
    print("-" * 60)
    print(result['pages'][0]['text'][:300] + "...")
    print("-" * 60)
else:
    print(f"❌ PDF file not found: {pdf_path}")

print("\n")

📄 File: attention_paper.pdf
📊 Total Pages: 15
✍️  Author: 
📖 Title: 

📝 First page preview (first 300 characters):
------------------------------------------------------------
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par...
------------------------------------------------------------




Why use low-level PyPDF?
- More control over how text is extracted
- Can access PDF metadata (author, title, creation date)
- Can handle special cases (encrypted PDFs, forms, etc.)
- Good for understanding what's happening under the hood

Limitations:
- More code to write
- Need to manually handle page splitting
- Doesn't integrate easily with RAG frameworks

In [ ]:
# method 2 - high level- using langchain's PyPDfLoader (PREFERRED METHOD FOR RAG PIPELINE)
def load_pdf_with_langchain(pdf_path: str):
  """
  preferred because :
  returnes document objects (standard format for langchain)
  automatically handles page splitting
  preserves metadata in a consistent format
  works seamlessly with other Langchain components

  Returns:
  List of document objects ( one per page )
  """
  loader = PyPDFLoader(pdf_path)
  documents = loader.load()
  return documents

In [ ]:
# testing this method
if os.path.exists(pdf_path):
    documents = load_pdf_with_langchain(pdf_path)

    print(f"📄 Loaded {len(documents)} pages as Document objects")
    print(f"\n🔍 Examining the first Document object:")
    print("-" * 60)

    first_doc = documents[0]

    print(f"Type: {type(first_doc)}")
    print(f"\nMetadata: {first_doc.metadata}")
    print(f"\nContent preview (first 300 chars):")
    print(first_doc.page_content[:300] + "...")
    print("-" * 60)

print("\n")

📄 Loaded 15 pages as Document objects

🔍 Examining the first Document object:
------------------------------------------------------------
Type: <class 'langchain_core.documents.base.Document'>

Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'sample_pdfs/attention_paper.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}

Content preview (first 300 chars):
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par...
-----------------------

What is Document object?
- Standard data structure in Langchain
- Has two main attributes:
   1. page_content: The actual text (string)
   2. metadata: Dictionary with info (page number, source file, etc.)

Why is this better for RAG?
 - Consistent format across different loaders (PDF, Word, HTML, etc.)
 - Metadata travels with the content through the pipeline
 - Easy to track where information came from (important for citations!)


In [ ]:
# load multiple PDFs from a directory
def load_all_pdfs_from_directory(directory_path: str) -> List:
  all_documents = []

  # get all PDF files in the directory
  pdf_files = [f for f in os.listdir(directory_path) if f.endswith('.pdf')]

  print(f"Found {len(pdf_files)} PDF file(s) in {directory_path}")
  print()

  for pdf_file in pdf_files:
      pdf_path = os.path.join(directory_path, pdf_file)
      print(f"  Loading: {pdf_file}...")

      try:
          loader = PyPDFLoader(pdf_path)
          documents = loader.load()
          all_documents.extend(documents)
          print(f"Loaded {len(documents)} pages")
      except Exception as e:
          print(f"Error loading {pdf_file}: {e}")

  return all_documents

In [ ]:
# Test it
all_docs = load_all_pdfs_from_directory("sample_pdfs")

Found 1 PDF file(s) in sample_pdfs

  Loading: attention_paper.pdf...
Loaded 15 pages
